# 🔍 DVF — Diagnostic complet des données

**Objectif :** comprendre précisément ce qu'on a avant de toucher au modèle.  
Chaque section produit des chiffres et graphiques concrets. Tu les captures et on adapte le pipeline ensemble.

**Plan :**
1. Chargement & aperçu brut
2. Volumétrie & couverture temporelle
3. Valeurs manquantes — carte complète
4. Distribution des variables clés
5. Détection des aberrants (prix, surface, prix/m²)
6. Géographie : couverture par département
7. Analyse de la cible : log(prix_m²)
8. Corrélations features → cible
9. Résidus du modèle actuel (si entraîné)
10. Importance des variables XGBoost
11. Analyse des erreurs par segment
12. Conclusions & checklist

In [ ]:
import sys
from pathlib import Path

# Chemin racine du projet
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display, HTML

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

FIGSIZE = (14, 5)
print('Imports OK')

In [ ]:
from src.config import RAW_PARQUET, CLEAN_PARQUET, FEATURES_PARQUET, MODEL_PATH, ENCODERS_PATH, FEATURE_COLUMNS, TARGET

# ── Chargement selon ce qui est disponible ─────────────────────────────────
def load_best_available():
    if FEATURES_PARQUET.exists():
        print(f'✅ Chargement features : {FEATURES_PARQUET}')
        return pd.read_parquet(FEATURES_PARQUET), 'features'
    if CLEAN_PARQUET.exists():
        print(f'⚠️  Features absentes — chargement nettoyé : {CLEAN_PARQUET}')
        return pd.read_parquet(CLEAN_PARQUET), 'clean'
    if RAW_PARQUET.exists():
        print(f'⚠️  Données nettoyées absentes — chargement brut : {RAW_PARQUET}')
        return pd.read_parquet(RAW_PARQUET), 'raw'
    raise FileNotFoundError(
        'Aucun fichier de données trouvé dans data/processed/.\n'
        'Lance : run.bat download   (puis run.bat clean si tu veux les données nettoyées)'
    )

df, DATA_STAGE = load_best_available()
print(f'Stage : {DATA_STAGE} | Shape : {df.shape}')

---
## 1 · Aperçu brut

In [ ]:
print(f'Lignes    : {len(df):>12,}')
print(f'Colonnes  : {df.shape[1]:>12}')
print(f'Mémoire   : {df.memory_usage(deep=True).sum() / 1e6:>10.1f} Mo')
df.head(3)

In [ ]:
df.dtypes.to_frame('dtype').T

In [ ]:
df.describe(include='all').T

---
## 2 · Volumétrie & couverture temporelle

In [ ]:
df['date_mutation'] = pd.to_datetime(df['date_mutation'], errors='coerce')

print(f"Plage temporelle : {df['date_mutation'].min().date()} → {df['date_mutation'].max().date()}")
print(f"Années couvertes : {sorted(df['date_mutation'].dt.year.dropna().unique().tolist())}")

by_month = df.set_index('date_mutation').resample('ME').size().rename('nb_transactions')

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)

by_month.plot(ax=axes[0])
axes[0].set_title('Transactions par mois')
axes[0].set_ylabel('Nombre')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

by_year = df['date_mutation'].dt.year.value_counts().sort_index()
by_year.plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Transactions par année')
axes[1].set_ylabel('Nombre')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.show()

print('\n📸 CAPTURE : volume par mois + année')

In [ ]:
# Répartition par type de bien
if 'type_local' in df.columns:
    type_counts = df['type_local'].value_counts()
    display(type_counts.to_frame('count').assign(pct=lambda x: (100*x['count']/x['count'].sum()).round(1)))
    
    fig, ax = plt.subplots(figsize=(7, 4))
    type_counts.plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title('Répartition par type de bien')
    ax.set_ylabel('Nombre de transactions')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()
    print('\n📸 CAPTURE : répartition type de bien')

---
## 3 · Valeurs manquantes — carte complète

In [ ]:
missing = (
    df.isnull().sum()
    .to_frame('n_missing')
    .assign(pct_missing=lambda x: (100 * x['n_missing'] / len(df)).round(2))
    .sort_values('pct_missing', ascending=False)
)
missing = missing[missing['n_missing'] > 0]
print(f'Colonnes avec valeurs manquantes : {len(missing)}')
display(missing)

In [ ]:
if len(missing) > 0:
    fig, ax = plt.subplots(figsize=(10, max(4, len(missing) * 0.4)))
    missing['pct_missing'].sort_values().plot(kind='barh', ax=ax, color='salmon')
    ax.set_xlabel('% manquant')
    ax.set_title('Taux de valeurs manquantes par colonne')
    ax.axvline(5, color='red', linestyle='--', alpha=0.5, label='5%')
    ax.axvline(50, color='darkred', linestyle='--', alpha=0.5, label='50%')
    ax.legend()
    plt.tight_layout()
    plt.show()
    print('\n📸 CAPTURE : carte des valeurs manquantes')

---
## 4 · Distribution des variables clés

In [ ]:
# Prix
prix_col = 'valeur_fonciere'
if prix_col in df.columns:
    prix = df[prix_col].dropna()
    print(f'=== {prix_col} ===')
    print(prix.describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]))
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].hist(prix.clip(0, prix.quantile(0.99)), bins=100, color='steelblue', edgecolor='none')
    axes[0].set_title('Distribution prix (clip 99e pct)')
    axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k€'))
    
    axes[1].hist(np.log1p(prix), bins=100, color='teal', edgecolor='none')
    axes[1].set_title('Distribution log(prix)')
    
    # Boxplot par type
    if 'type_local' in df.columns:
        for t, grp in df.groupby('type_local'):
            axes[2].boxplot(
                grp[prix_col].clip(0, grp[prix_col].quantile(0.95)).dropna(),
                positions=[list(df['type_local'].unique()).index(t)],
                widths=0.6, patch_artist=True,
            )
        axes[2].set_xticks(range(df['type_local'].nunique()))
        axes[2].set_xticklabels(df['type_local'].unique(), rotation=20)
        axes[2].set_title('Prix par type (clip 95e pct)')
        axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k€'))
    
    plt.tight_layout()
    plt.show()
    print('\n📸 CAPTURE : distributions du prix')

In [ ]:
# Surface
surf_col = 'surface_reelle_bati'
if surf_col in df.columns:
    surf = df[surf_col].dropna()
    print(f'=== {surf_col} ===')
    print(surf.describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]))
    
    fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)
    axes[0].hist(surf.clip(0, surf.quantile(0.99)), bins=80, color='steelblue', edgecolor='none')
    axes[0].set_title('Distribution surface (clip 99e pct)')
    axes[0].set_xlabel('m²')
    
    if 'type_local' in df.columns:
        for t in df['type_local'].unique():
            mask = df['type_local'] == t
            axes[1].hist(df.loc[mask, surf_col].clip(0, 300).dropna(),
                        bins=60, alpha=0.6, label=t, edgecolor='none')
        axes[1].set_title('Surface par type')
        axes[1].set_xlabel('m²')
        axes[1].legend()
    
    plt.tight_layout()
    plt.show()
    print('\n📸 CAPTURE : distributions de la surface')

In [ ]:
# Nombre de pièces
pieces_col = 'nombre_pieces_principales'
if pieces_col in df.columns:
    pieces = df[pieces_col].dropna()
    vc = pieces.value_counts().sort_index()
    display(vc.to_frame('count').assign(pct=lambda x: (100*x['count']/x['count'].sum()).round(1)).head(15))
    
    fig, ax = plt.subplots(figsize=(10, 4))
    vc.head(15).plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title('Répartition par nombre de pièces')
    ax.set_ylabel('Nombre de transactions')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    plt.tight_layout()
    plt.show()
    print('\n📸 CAPTURE : répartition nombre de pièces')

---
## 5 · Détection des aberrants

In [ ]:
# Calcul du prix au m² si pas encore fait
if 'prix_m2' not in df.columns and 'valeur_fonciere' in df.columns and 'surface_reelle_bati' in df.columns:
    df['prix_m2'] = df['valeur_fonciere'] / df['surface_reelle_bati']

if 'prix_m2' in df.columns:
    pm2 = df['prix_m2'].dropna().replace([np.inf, -np.inf], np.nan).dropna()
    
    print('=== Prix au m² — percentiles ===')
    print(pm2.describe(percentiles=[.01,.05,.1,.25,.5,.75,.9,.95,.99,.999]))
    
    # Seuils aberrants
    thresholds = [200, 500, 1000, 5000, 10000, 15000, 20000, 25000, 30000]
    print('\n=== % transactions en dehors de différents seuils ===')
    for lo, hi in [(200, 25000), (500, 20000), (500, 15000)]:
        pct_out = 100 * ((pm2 < lo) | (pm2 > hi)).sum() / len(pm2)
        print(f'  [{lo:>6,} – {hi:>6,} €/m²]  exclus : {pct_out:.2f}%')
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    
    axes[0].hist(pm2.clip(0, pm2.quantile(0.99)), bins=100, color='steelblue', edgecolor='none')
    axes[0].set_title('Distribution prix/m² (clip 99e pct)')
    axes[0].set_xlabel('€/m²')
    axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    
    axes[1].hist(np.log1p(pm2), bins=100, color='teal', edgecolor='none')
    axes[1].set_title('Distribution log(prix/m²)')
    axes[1].set_xlabel('log(€/m²)')
    
    # Scatter prix vs surface
    if 'surface_reelle_bati' in df.columns:
        sample = df.dropna(subset=['prix_m2', 'surface_reelle_bati']).sample(
            min(30_000, len(df)), random_state=42
        )
        axes[2].scatter(sample['surface_reelle_bati'], sample['prix_m2'],
                       alpha=0.05, s=2, color='navy')
        axes[2].set_xlim(0, 300)
        axes[2].set_ylim(0, pm2.quantile(0.99))
        axes[2].set_title('Prix/m² vs Surface')
        axes[2].set_xlabel('Surface (m²)')
        axes[2].set_ylabel('Prix/m² (€)')
    
    plt.tight_layout()
    plt.show()
    print('\n📸 CAPTURE : aberrants prix/m²')

In [ ]:
# Aberrants extrêmes — les lignes les plus suspectes
if 'prix_m2' in df.columns:
    print('=== 10 prix/m² les plus ÉLEVÉS ===')
    display(
        df.nlargest(10, 'prix_m2')[[
            c for c in ['valeur_fonciere','surface_reelle_bati','prix_m2',
                        'type_local','code_departement','nom_commune','date_mutation']
            if c in df.columns
        ]]
    )
    
    print('\n=== 10 prix/m² les plus BAS (> 0) ===')
    display(
        df[df['prix_m2'] > 0].nsmallest(10, 'prix_m2')[[
            c for c in ['valeur_fonciere','surface_reelle_bati','prix_m2',
                        'type_local','code_departement','nom_commune','date_mutation']
            if c in df.columns
        ]]
    )
    print('\n📸 CAPTURE : lignes aberrantes extrêmes')

---
## 6 · Géographie : couverture par département

In [ ]:
if 'code_departement' in df.columns and 'prix_m2' in df.columns:
    by_dept = (
        df.groupby('code_departement')
        .agg(
            n=('prix_m2', 'count'),
            prix_median=('prix_m2', 'median'),
            prix_mean=('prix_m2', 'mean'),
            prix_std=('prix_m2', 'std'),
            prix_p10=('prix_m2', lambda x: x.quantile(0.1)),
            prix_p90=('prix_m2', lambda x: x.quantile(0.9)),
        )
        .sort_values('prix_median', ascending=False)
        .reset_index()
    )
    by_dept['cv'] = (by_dept['prix_std'] / by_dept['prix_mean'] * 100).round(1)  # coeff de variation
    
    print(f'Départements couverts : {len(by_dept)}')
    display(by_dept.head(20))
    
    fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)
    
    # Top 30 par volume
    top_vol = by_dept.nlargest(30, 'n')
    axes[0].barh(top_vol['code_departement'], top_vol['n'], color='steelblue')
    axes[0].set_title('Top 30 départements par volume')
    axes[0].set_xlabel('Transactions')
    axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    
    # Top 30 par prix médian
    top_price = by_dept.head(30)
    bars = axes[1].barh(top_price['code_departement'][::-1],
                        top_price['prix_median'][::-1], color='coral')
    axes[1].set_title('Top 30 départements par prix médian/m²')
    axes[1].set_xlabel('€/m²')
    axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    
    plt.tight_layout()
    plt.show()
    print('\n📸 CAPTURE : couverture géographique')

In [ ]:
# Coefficient de variation par département (hétérogénéité intra-dept)
if 'by_dept' in dir():
    print('=== Hétérogénéité intra-département (Coeff. Variation) ===')
    print('(Élevé = le modèle aura du mal car trop de variance à expliquer)')
    display(by_dept[['code_departement','n','prix_median','cv']].sort_values('cv', ascending=False).head(20))
    
    fig, ax = plt.subplots(figsize=(10, 4))
    by_dept_sorted = by_dept.sort_values('cv', ascending=True)
    color = ['crimson' if cv > 80 else 'steelblue' for cv in by_dept_sorted['cv']]
    ax.barh(by_dept_sorted['code_departement'], by_dept_sorted['cv'], color=color)
    ax.axvline(80, color='red', linestyle='--', alpha=0.7, label='Seuil alerte (80%)')
    ax.set_title('Coefficient de variation du prix/m² par département\n(rouge = très hétérogène)')
    ax.set_xlabel('CV (%)')
    ax.legend()
    plt.tight_layout()
    plt.show()
    print('\n📸 CAPTURE : hétérogénéité intra-département')

---
## 7 · Analyse de la cible : log(prix_m²)

In [ ]:
if 'prix_m2' in df.columns:
    pm2 = df['prix_m2'].replace([np.inf, -np.inf], np.nan).dropna()
    log_pm2 = np.log1p(pm2)
    
    from scipy import stats
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    
    # Raw
    axes[0].hist(pm2.clip(0, pm2.quantile(0.995)), bins=100, color='steelblue', density=True, edgecolor='none')
    axes[0].set_title(f'Prix/m² brut\nskew={pm2.skew():.2f}, kurt={pm2.kurtosis():.2f}')
    axes[0].set_xlabel('€/m²')
    
    # Log
    axes[1].hist(log_pm2, bins=100, color='teal', density=True, edgecolor='none')
    x = np.linspace(log_pm2.min(), log_pm2.max(), 200)
    axes[1].plot(x, stats.norm.pdf(x, log_pm2.mean(), log_pm2.std()), 'r-', lw=2, label='Normale')
    axes[1].set_title(f'log(prix/m²)\nskew={log_pm2.skew():.2f}, kurt={log_pm2.kurtosis():.2f}')
    axes[1].set_xlabel('log(€/m²)')
    axes[1].legend()
    
    # QQ-plot
    stats.probplot(log_pm2, dist='norm', plot=axes[2])
    axes[2].set_title('QQ-plot log(prix/m²) vs Normale')
    
    plt.tight_layout()
    plt.show()
    
    print(f'Skewness prix brut   : {pm2.skew():.3f}  (idéal proche de 0)')
    print(f'Skewness log(prix)   : {log_pm2.skew():.3f}  (idéal proche de 0)')
    print('\n📸 CAPTURE : distribution de la cible')

---
## 8 · Corrélations features → cible

In [ ]:
if 'log_prix_m2' not in df.columns and 'prix_m2' in df.columns:
    df['log_prix_m2'] = np.log1p(df['prix_m2'].replace([np.inf, -np.inf], np.nan))

num_cols = df.select_dtypes(include='number').columns.tolist()
if 'log_prix_m2' in df.columns:
    corr_target = (
        df[num_cols].corr()['log_prix_m2']
        .drop('log_prix_m2', errors='ignore')
        .dropna()
        .sort_values(key=abs, ascending=False)
    )
    
    print('=== Corrélations (Pearson) avec log(prix_m²) ===')
    display(corr_target.head(25).to_frame('corr_pearson').round(3))
    
    fig, ax = plt.subplots(figsize=(10, max(5, len(corr_target.head(20)) * 0.4)))
    colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in corr_target.head(20)]
    corr_target.head(20).plot(kind='barh', ax=ax, color=colors)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title('Top 20 corrélations avec log(prix/m²)')
    ax.set_xlabel('Corrélation de Pearson')
    plt.tight_layout()
    plt.show()
    print('\n📸 CAPTURE : corrélations avec la cible')

In [ ]:
# Heatmap des corrélations entre features du modèle
feat_present = [c for c in FEATURE_COLUMNS if c in df.columns]
if len(feat_present) >= 3 and 'log_prix_m2' in df.columns:
    corr_matrix = df[feat_present + ['log_prix_m2']].corr()
    
    fig, ax = plt.subplots(figsize=(12, 10))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
    sns.heatmap(
        corr_matrix, mask=~mask, annot=True, fmt='.2f',
        cmap='RdBu_r', center=0, ax=ax,
        annot_kws={'size': 8},
    )
    ax.set_title('Matrice de corrélation — features du modèle')
    plt.tight_layout()
    plt.show()
    print('\n📸 CAPTURE : heatmap corrélations')

---
## 9 · Résidus du modèle actuel

In [ ]:
if not MODEL_PATH.exists():
    print('⚠️  Modèle non entraîné — section ignorée.')
    print('   Lance : run.bat train')
else:
    import joblib
    from sklearn.metrics import mean_squared_error, r2_score
    
    model = joblib.load(MODEL_PATH)
    feat_present = [c for c in FEATURE_COLUMNS if c in df.columns]
    missing_feat = [c for c in FEATURE_COLUMNS if c not in df.columns]
    
    if missing_feat:
        print(f'⚠️  Features manquantes pour le modèle : {missing_feat}')
    
    df_model = df[feat_present + ['log_prix_m2']].dropna()
    X = df_model[feat_present]
    y_log = df_model['log_prix_m2']
    
    preds_log = model.predict(X)
    y_true = np.expm1(y_log)
    y_pred = np.expm1(preds_log)
    residuals = y_true - y_pred
    pct_error = (residuals / y_true) * 100
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs(pct_error))
    
    print(f'RMSE  : {rmse:>10,.1f} €/m²')
    print(f'R²    : {r2:>10.4f}')
    print(f'MAPE  : {mape:>10.2f} %')
    print(f'MAE   : {np.mean(np.abs(residuals)):>10,.1f} €/m²')
    print(f'Médiane |erreur %| : {np.median(np.abs(pct_error)):>6.2f} %')

In [ ]:
if MODEL_PATH.exists() and 'residuals' in dir():
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Résidus vs prédictions
    axes[0,0].scatter(y_pred, residuals, alpha=0.03, s=2, color='navy')
    axes[0,0].axhline(0, color='red', linewidth=1)
    axes[0,0].set_xlabel('Prédiction (€/m²)')
    axes[0,0].set_ylabel('Résidu (€/m²)')
    axes[0,0].set_title('Résidus vs Prédictions')
    axes[0,0].set_xlim(0, np.percentile(y_pred, 99))
    axes[0,0].set_ylim(np.percentile(residuals, 1), np.percentile(residuals, 99))
    
    # Distribution des résidus
    axes[0,1].hist(residuals.clip(*np.percentile(residuals, [1, 99])), bins=100,
                  color='steelblue', edgecolor='none')
    axes[0,1].axvline(0, color='red')
    axes[0,1].set_title(f'Distribution résidus\nskew={pd.Series(residuals).skew():.2f}')
    axes[0,1].set_xlabel('Résidu (€/m²)')
    
    # Prédictions vs vraies valeurs
    lim = np.percentile(np.concatenate([y_true, y_pred]), 99)
    axes[1,0].scatter(y_true, y_pred, alpha=0.03, s=2, color='teal')
    axes[1,0].plot([0, lim], [0, lim], 'r-', lw=1.5, label='Parfait')
    axes[1,0].set_xlim(0, lim)
    axes[1,0].set_ylim(0, lim)
    axes[1,0].set_xlabel('Vrai prix/m²')
    axes[1,0].set_ylabel('Prédit prix/m²')
    axes[1,0].set_title('Vrai vs Prédit')
    axes[1,0].legend()
    
    # Distribution erreur %
    axes[1,1].hist(pct_error.clip(-100, 100), bins=100, color='coral', edgecolor='none')
    axes[1,1].axvline(0, color='red')
    axes[1,1].axvline(-20, color='orange', linestyle='--', alpha=0.7, label='±20%')
    axes[1,1].axvline(20, color='orange', linestyle='--', alpha=0.7)
    axes[1,1].set_title(f'Erreur relative\nMAPE={mape:.1f}%')
    axes[1,1].set_xlabel('Erreur (%)')
    axes[1,1].legend()
    
    plt.tight_layout()
    plt.show()
    print('\n📸 CAPTURE : analyse complète des résidus')

---
## 10 · Importance des variables XGBoost

In [ ]:
if MODEL_PATH.exists():
    import joblib
    model = joblib.load(MODEL_PATH)
    feat_present = [c for c in FEATURE_COLUMNS if c in df.columns]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    importance_types = ['weight', 'gain', 'cover']
    
    for ax, imp_type in zip(axes, importance_types):
        try:
            imp = model.get_booster().get_score(importance_type=imp_type)
            imp_series = pd.Series(imp).reindex(
                [f'f{i}' for i in range(len(feat_present))]
            )
            imp_series.index = feat_present[:len(imp_series)]
        except:
            imp_series = pd.Series(model.feature_importances_, index=feat_present)
        
        imp_series = imp_series.dropna().sort_values(ascending=True)
        imp_series.plot(kind='barh', ax=ax, color='steelblue')
        ax.set_title(f'Importance : {imp_type}')
        ax.set_xlabel(imp_type)
    
    plt.tight_layout()
    plt.show()
    print('\n📸 CAPTURE : importance des variables (3 méthodes)')
else:
    print('⚠️  Modèle non entraîné — section ignorée.')

---
## 11 · Analyse des erreurs par segment

In [ ]:
if MODEL_PATH.exists() and 'pct_error' in dir():
    df_err = df_model.copy()
    df_err['y_true']    = y_true
    df_err['y_pred']    = y_pred
    df_err['residual']  = residuals
    df_err['pct_error'] = pct_error
    df_err['abs_pct']   = np.abs(pct_error)
    
    def segment_metrics(group_col):
        if group_col not in df_err.columns:
            return None
        return (
            df_err.groupby(group_col)
            .agg(
                n=('y_true', 'count'),
                rmse=('residual', lambda x: np.sqrt((x**2).mean())),
                mape=('abs_pct', 'mean'),
                biais=('pct_error', 'mean'),
            )
            .round(2)
            .sort_values('mape', ascending=False)
        )
    
    # Par type de bien
    if 'type_local' in df_err.columns:
        print('=== Erreurs par type de bien ===')
        display(segment_metrics('type_local'))
    
    # Par département (top 20 pires)
    if 'code_departement' in df_err.columns:
        print('\n=== Top 20 départements avec le plus d\'erreur (MAPE) ===')
        by_dept_err = segment_metrics('code_departement')
        display(by_dept_err[by_dept_err['n'] >= 100].head(20))
    
    # Par tranche de prix réel
    df_err['tranche_prix'] = pd.cut(
        df_err['y_true'],
        bins=[0, 2000, 4000, 6000, 8000, 12000, 20000, 99999],
        labels=['<2k', '2–4k', '4–6k', '6–8k', '8–12k', '12–20k', '>20k'],
    )
    print('\n=== Erreurs par tranche de prix/m² ===')
    display(segment_metrics('tranche_prix'))
    
    # Visualisation biais par département
    if 'code_departement' in df_err.columns:
        dept_bias = segment_metrics('code_departement')
        dept_bias = dept_bias[dept_bias['n'] >= 200].sort_values('biais')
        
        fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)
        
        colors = ['#e74c3c' if b > 0 else '#3498db' for b in dept_bias['biais']]
        dept_bias['biais'].plot(kind='barh', ax=axes[0], color=colors)
        axes[0].axvline(0, color='black', linewidth=0.8)
        axes[0].set_title('Biais moyen par département (% erreur)\nRouge = sous-estimation, Bleu = surestimation')
        axes[0].set_xlabel('Biais moyen (%)')
        
        dept_mape = dept_bias.sort_values('mape', ascending=True)
        dept_mape['mape'].plot(kind='barh', ax=axes[1], color='coral')
        axes[1].axvline(20, color='red', linestyle='--', alpha=0.7, label='20%')
        axes[1].set_title('MAPE par département')
        axes[1].set_xlabel('MAPE (%)')
        axes[1].legend()
        
        plt.tight_layout()
        plt.show()
        print('\n📸 CAPTURE : analyse erreurs par segment')

In [ ]:
# Les pires prédictions individuelles
if MODEL_PATH.exists() and 'df_err' in dir():
    worst = df_err.nlargest(20, 'abs_pct')
    print('=== 20 pires prédictions (erreur % absolue) ===')
    display(worst[[
        c for c in ['y_true','y_pred','residual','pct_error',
                    'type_local','code_departement','surface_reelle_bati']
        if c in worst.columns
    ]].round(1))
    print('\n📸 CAPTURE : pires prédictions individuelles')

---
## 12 · Analyse saisonnalité & tendance

In [ ]:
if 'prix_m2' in df.columns and 'date_mutation' in df.columns:
    df_time = df.copy()
    df_time['annee'] = df_time['date_mutation'].dt.year
    df_time['mois']  = df_time['date_mutation'].dt.month
    df_time['trimestre'] = df_time['date_mutation'].dt.quarter
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Prix médian par mois (saisonnalité)
    sais = df_time.groupby('mois')['prix_m2'].median()
    sais.plot(ax=axes[0], marker='o', color='steelblue')
    axes[0].set_title('Saisonnalité : prix médian/m² par mois')
    axes[0].set_xlabel('Mois')
    axes[0].set_ylabel('€/m²')
    axes[0].set_xticks(range(1, 13))
    axes[0].set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    
    # Prix médian par trimestre
    trim = df_time.groupby(['annee','trimestre'])['prix_m2'].median().reset_index()
    trim['periode'] = trim['annee'].astype(str) + '-Q' + trim['trimestre'].astype(str)
    axes[1].plot(trim['periode'], trim['prix_m2'], marker='o', color='teal')
    axes[1].set_title('Prix médian/m² par trimestre')
    axes[1].set_ylabel('€/m²')
    axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    # Nb transactions par trimestre
    trim_vol = df_time.groupby(['annee','trimestre']).size().reset_index(name='n')
    trim_vol['periode'] = trim_vol['annee'].astype(str) + '-Q' + trim_vol['trimestre'].astype(str)
    axes[2].bar(trim_vol['periode'], trim_vol['n'], color='coral')
    axes[2].set_title('Volume transactions par trimestre')
    axes[2].set_ylabel('Transactions')
    axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    plt.setp(axes[2].xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()
    print('\n📸 CAPTURE : saisonnalité et tendance')

---
## 13 · Coordonnées GPS — couverture

In [ ]:
for coord_col in ('latitude', 'longitude'):
    if coord_col in df.columns:
        n_total = len(df)
        n_ok = df[coord_col].notna().sum()
        print(f'{coord_col:15s} : {n_ok:>8,} / {n_total:>8,}  ({100*n_ok/n_total:.1f}% couvert)')

if 'latitude' in df.columns and 'longitude' in df.columns:
    gps_ok = df[df['latitude'].notna() & df['longitude'].notna()]
    sample_gps = gps_ok.sample(min(50_000, len(gps_ok)), random_state=42)
    
    # Bornes France métropolitaine
    france = sample_gps[
        (sample_gps['latitude'].between(41, 51)) &
        (sample_gps['longitude'].between(-5, 10))
    ]
    hors = len(sample_gps) - len(france)
    print(f'\nCoordonnées hors France métro : {hors} ({100*hors/len(sample_gps):.1f}%)')
    
    fig, ax = plt.subplots(figsize=(8, 8))
    if 'prix_m2' in france.columns:
        sc = ax.scatter(
            france['longitude'], france['latitude'],
            c=np.log1p(france['prix_m2']),
            cmap='RdYlGn_r', s=1, alpha=0.3
        )
        plt.colorbar(sc, ax=ax, label='log(prix/m²)')
    else:
        ax.scatter(france['longitude'], france['latitude'], s=1, alpha=0.3)
    ax.set_title(f'Couverture GPS — {len(france):,} points'.replace(',', '\u202f'))
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    plt.tight_layout()
    plt.show()
    print('\n📸 CAPTURE : couverture GPS')

---
## 14 · Checklist diagnostique

In [ ]:
print('=' * 60)
print('CHECKLIST DIAGNOSTIQUE DVF')
print('=' * 60)

checks = []

# Volume
n = len(df)
ok = n >= 200_000
checks.append(('Volume ≥ 200k transactions', ok, f'{n:,}'))

# Prix/m² : taux aberrants
if 'prix_m2' in df.columns:
    pm2 = df['prix_m2'].dropna().replace([np.inf,-np.inf], np.nan).dropna()
    pct_bad = 100 * ((pm2 < 500) | (pm2 > 20000)).sum() / len(pm2)
    ok = pct_bad < 5
    checks.append(('Aberrants prix/m² < 5%', ok, f'{pct_bad:.2f}%'))

# Surface : taux manquants
if 'surface_reelle_bati' in df.columns:
    pct_miss = 100 * df['surface_reelle_bati'].isna().sum() / n
    ok = pct_miss < 5
    checks.append(('Surface manquante < 5%', ok, f'{pct_miss:.2f}%'))

# GPS couverture
if 'latitude' in df.columns:
    pct_gps = 100 * df['latitude'].notna().sum() / n
    ok = pct_gps > 70
    checks.append(('GPS couverture > 70%', ok, f'{pct_gps:.1f}%'))

# Skewness cible
if 'log_prix_m2' in df.columns:
    sk = df['log_prix_m2'].dropna().skew()
    ok = abs(sk) < 1.0
    checks.append(('Skewness log(prix/m²) < 1', ok, f'{sk:.3f}'))

# Corrélation revenu si dispo
if 'revenu_median' in df.columns and 'log_prix_m2' in df.columns:
    corr_rev = df[['revenu_median','log_prix_m2']].corr().iloc[0,1]
    ok = abs(corr_rev) > 0.3
    checks.append(('Corrélation revenu_median > 0.3', ok, f'{corr_rev:.3f}'))

# Modèle R²
if MODEL_PATH.exists() and 'r2' in dir():
    ok = r2 > 0.80
    checks.append(('R² modèle > 0.80', ok, f'{r2:.3f}'))

for label, ok, val in checks:
    icon = '✅' if ok else '❌'
    print(f'  {icon}  {label:<45} {val}')

print()
print('📸 CAPTURE : cette checklist entière')

In [ ]:
# Résumé final à copier-coller dans le chat
print('\n' + '=' * 60)
print('RÉSUMÉ À COPIER DANS LE CHAT')
print('=' * 60)
print(f'Lignes           : {len(df):,}')
print(f'Stage pipeline   : {DATA_STAGE}')
if 'date_mutation' in df.columns:
    print(f'Période          : {df["date_mutation"].min().date()} → {df["date_mutation"].max().date()}')
if 'type_local' in df.columns:
    print(f'Types de biens   : {df["type_local"].value_counts().to_dict()}')
if 'code_departement' in df.columns:
    print(f'Départements     : {df["code_departement"].nunique()}')
if 'prix_m2' in df.columns:
    pm2 = df['prix_m2'].dropna().replace([np.inf,-np.inf],np.nan).dropna()
    print(f'Prix/m² médian   : {pm2.median():,.0f} €')
    print(f'Prix/m² P1-P99   : {pm2.quantile(0.01):,.0f} – {pm2.quantile(0.99):,.0f} €')
if 'latitude' in df.columns:
    print(f'Couverture GPS   : {100*df["latitude"].notna().mean():.1f}%')
if MODEL_PATH.exists() and 'r2' in dir():
    print(f'R² modèle actuel : {r2:.4f}')
    print(f'RMSE             : {rmse:,.1f} €/m²')
    print(f'MAPE             : {mape:.2f}%')